
# C7-cnn-transfer — Practice p15

**Type:** integrative (parts consume earlier results) · **Difficulty:** core · **Concepts:** resnet architecture, bottleneck blocks

One network, every reading register.

**(a) Shape trace, by hand.**
For a `(1, 3, 192, 192)` input, fill `shapes_hand` with plain tuples
for the tensor *after* each listed child.
Work from the output-size formula and the stage constants.
**Running the model or any submodule is banned in this part (zero
points)** — the verification cell in (b) is the only place forwards
happen.

**(b) Verification.**
The provided loop replays the children on a real `(1, 3, 192, 192)`
batch and compares against your (a); `all_match` must be `True`.

**(c) Block census, from inspection.**
`n_blocks` — the per-stage block counts as a 4-tuple;
`n_body_convs` — total convolutions inside all bottleneck blocks
(3 per block — arithmetic, not iteration);
`depth_50` — the count that names the network: body convs + the two
other weighted layers (plain `int`, arithmetic from `n_body_convs`).

**(d) Hand count.**
For `model.layer2[3]` (an interior block, $512 \to 128 \to 128 \to
512$): `hand_convs`, `hand_bn`, `hand_total` as int literals with
the arithmetic in comments.
**`numel` / `torchsummary` / `state_dict`-size reads — incl. `p.nelement()`,
`np.prod(p.shape)`, and every disguised element-count helper — are banned
in this cell (zero points).**

**(e) Check.**
`torch_total` — the `numel` sum over `model.layer2[3].parameters()`
(**allowed here**) — and `count_gap = abs(hand_total - torch_total)`,
which must be `0`.


In [ ]:
# Cache pin (course convention, plan 009): pretrained weights live in the repo's
# gitignored reference/cache/ -- resolve it from the repo root BEFORE importing torch.
import os, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
             if (p / "pyproject.toml").exists())
os.environ["TORCH_HOME"] = str(_root / "reference" / "cache" / "torch")

import torch
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights

# float32 register (course exception): pretrained resnet50 is a float32 artifact.
# No float64 default here; inputs are cast .to(torch.float32) at the model
# boundary; float comparisons state atol=1e-6 / rtol=1e-5.
SEED = 20260804

model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
model.eval()
assert next(model.parameters()).dtype == torch.float32


# (a) hand shape trace -- fill every value with a plain tuple
shapes_hand = {
    "conv1":   ...,   # YOUR ANSWER HERE
    "maxpool": ...,   # YOUR ANSWER HERE
    "layer1":  ...,   # YOUR ANSWER HERE
    "layer2":  ...,   # YOUR ANSWER HERE
    "layer3":  ...,   # YOUR ANSWER HERE
    "layer4":  ...,   # YOUR ANSWER HERE
    "avgpool": ...,   # YOUR ANSWER HERE
    "fc":      ...,   # YOUR ANSWER HERE
}


In [ ]:

# (b) verification -- provided; do not edit
torch.manual_seed(SEED)
x192 = torch.randn(1, 3, 192, 192).to(torch.float32)
seen = {}
cur = x192
with torch.inference_mode():
    for name, child in model.named_children():
        if name == "fc":
            cur = torch.flatten(cur, 1)
        cur = child(cur)
        if name in ("conv1", "maxpool", "layer1", "layer2",
                    "layer3", "layer4", "avgpool", "fc"):
            seen[name] = tuple(cur.shape)

all_match = ...   # YOUR CODE HERE -- compare seen against shapes_hand


In [ ]:

# (c) census from inspection
n_blocks = ...      # YOUR CODE HERE
n_body_convs = ...  # YOUR CODE HERE
depth_50 = ...      # YOUR CODE HERE


In [ ]:

# (d) hand count for model.layer2[3] -- numel banned in this cell
hand_convs = ...  # YOUR CODE HERE (int literal; arithmetic in a comment)
hand_bn = ...     # YOUR CODE HERE
hand_total = ...  # YOUR CODE HERE


In [ ]:

# (e) the check -- numel allowed here
torch_total = ...  # YOUR CODE HERE
count_gap = ...    # YOUR CODE HERE -- must be 0
